# 43 — Explainability: glassbox, and the black box beside it

**Reads:** `ml.train_matrix`, `ml.test_matrix`, `ml.test_raw`, and `@champion`
**Segment 4 of the session — hero moment ★1**

Ported from `07_ebm_explainability` and the SHAP sections of `05_modelling`.

### The argument this notebook has to make

Two models. One explains itself; the other needs a second model to explain it.

* **The EBM is additive by construction.** Its prediction *is* the sum of its shape
  functions. When it says a pre-op OKS of 16 contributed +0.42, that is not an estimate of
  what the model did — it is what the model did.
* **SHAP on the random forest is an approximation**, and a good one, but it is a separate
  model of the first model. Which raises a question you have to be able to answer in front
  of a regulator: *how do you know the explanation is faithful?*

For a glassbox model that question does not arise. That is the entire case for accepting
slightly lower accuracy in a high-risk clinical setting, and this notebook is where you
make it visible rather than assert it.

## 1. Load

In [ ]:
import mlflow, mlflow.sklearn
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from mlflow import MlflowClient

EXPERIMENT       = "knee-poor-outcome"
REGISTERED_MODEL = "knee-poor-outcome-ebm"
CHAMPION_LABEL   = "champion"   # a version tag, not an MLflow alias -- Fabric has no alias API

# Paste the id printed by 30_gold_features §8. Leave as None to pick the highest-risk
# Margaret-like patient in the test set instead.
MARGARET_EPISODE_ID = None

# --- champion resolution ----------------------------------------------------
# Fabric's MLflow registry does not implement the alias endpoints
# (/api/2.0/mlflow/registered-models/alias -> 404), so `@champion` URIs and
# get_model_version_by_alias() are unavailable. Version tags work, so the
# champion pointer is the tag champion='true'. This helper tries the alias
# first so the code stays correct on a tenant where aliases do exist.
def resolve_champion(client, name):
    try:
        return client.get_model_version_by_alias(name, "champion")
    except Exception:
        pass
    versions = list(client.search_model_versions(f"name='{name}'"))
    if not versions:
        raise RuntimeError(f"No versions registered for {name} -- run 40_train_register first.")
    champions = [v for v in versions if v.tags.get("champion") == "true"]
    if champions:
        return max(champions, key=lambda v: int(v.version))
    latest = max(versions, key=lambda v: int(v.version))
    print(f"WARNING: no champion tag found; falling back to latest version {latest.version}")
    return latest


mlflow.set_experiment(EXPERIMENT)
mlflow.autolog(disable=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

def load(table):
    pdf = spark.table(table).toPandas()
    y = pdf.pop("outcome_label").astype(int)
    return pdf, y

X_train, y_train = load("ml.train_matrix")
X_test,  y_test  = load("ml.test_matrix")
X_test   = X_test[X_train.columns]
raw_test = spark.table("ml.test_raw").toPandas()

mv    = resolve_champion(MlflowClient(), REGISTERED_MODEL)
model = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL}/{mv.version}")
y_prob = model.predict_proba(X_test)[:, 1]

print(f"{REGISTERED_MODEL} v{mv.version} | test {X_test.shape}")

In [ ]:
def underlying_ebm(m):
    """Reach through CalibratedClassifierCV to the fitted EBM."""
    if hasattr(m, "explain_global"):
        return m
    for cc in getattr(m, "calibrated_classifiers_", []):
        est = getattr(cc, "estimator", None) or getattr(cc, "base_estimator", None)
        if hasattr(est, "explain_global"):
            return est
    return None

ebm = underlying_ebm(model)
assert ebm is not None, "No EBM found inside the champion model — is @champion the EBM?"
print(f"EBM recovered: {len(ebm.term_names_)} terms "
      f"({sum(1 for t in ebm.term_features_ if len(t) > 1)} pairwise interactions)")

## 2. Global shape functions — the model, drawn

An EBM learns one function per feature: *given this value, how much does risk move?* The
prediction is the sum of those functions. Nothing else.

So the plots below are not a summary of the model. They **are** the model, and a clinician
can read them and disagree with them — which is exactly the "meaningful human oversight"
the AI Act asks for. A clinician cannot disagree with a random forest.

Watch for non-monotonic shapes. A linear model would force a straight line through these;
the EBM does not, and where the curve bends is usually where the clinical interest is.

In [ ]:
global_exp = ebm.explain_global()
term_names = ebm.term_names_

importances = ebm.term_importances()
order = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 6.5))
top = order[:20][::-1]
ax.barh([term_names[i] for i in top], [importances[i] for i in top], color="#4C78A8")
ax.set_xlabel("Mean absolute contribution to log-odds")
ax.set_title("EBM term importance — the whole model, ranked")
plt.tight_layout(); plt.show()

print("Top 10 terms:")
for rank, i in enumerate(order[:10], 1):
    kind = "interaction" if len(ebm.term_features_[i]) > 1 else "main effect"
    print(f"  {rank:2d}. {term_names[i]:<38s} {importances[i]:.4f}  ({kind})")

In [ ]:
# The shape functions for the six most important main effects.
MAIN = [i for i in order if len(ebm.term_features_[i]) == 1][:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
for ax, i in zip(axes.ravel(), MAIN):
    d = global_exp.data(i)
    x, yv = d["names"], d["scores"]

    if d.get("type") == "univariate" and len(x) == len(yv) + 1:
        centres = [(x[k] + x[k + 1]) / 2 for k in range(len(yv))]   # bin edges -> centres
        ax.step(centres, yv, where="mid", lw=2, color="#4C78A8")
        if "upper_bounds" in d and d["upper_bounds"] is not None:
            ax.fill_between(centres, d["lower_bounds"], d["upper_bounds"],
                            alpha=.2, color="#4C78A8", step="mid")
    else:
        ax.bar(range(len(yv)), yv, color="#4C78A8")
        ax.set_xticks(range(len(yv)))
        ax.set_xticklabels([str(v) for v in x], rotation=45, ha="right", fontsize=8)

    ax.axhline(0, color="#C44E52", lw=1, ls="--")
    ax.set_title(term_names[i], fontsize=10, fontweight="bold")
    ax.set_ylabel("→ risk of poor outcome")
    ax.grid(alpha=.25)

plt.tight_layout(); plt.show()
print("Above the red line increases risk, below it reduces risk. "
      "The shaded band is model uncertainty across the outer bags.")

## 3. Margaret — the local explanation

One patient. The bars sum, with the intercept, to her score. That is the arithmetic, not an
analogy — which is what makes it defensible when she asks *why*.

In [ ]:
if MARGARET_EPISODE_ID and "episode_id" in raw_test.columns:
    matches = raw_test.index[raw_test["episode_id"] == MARGARET_EPISODE_ID].tolist()
    idx = matches[0] if matches else None
    if idx is None:
        print(f"{MARGARET_EPISODE_ID} is not in the test split — picking a stand-in.")
else:
    idx = None

if idx is None:
    # A genuinely borderline patient: moderate risk, low pre-op score, comorbid.
    cand = pd.DataFrame({"i": range(len(y_prob)), "p": y_prob})
    cand = cand[(cand.p > 0.30) & (cand.p < 0.45)]
    if "comorbidity_count" in raw_test.columns and len(cand):
        cand = cand.assign(cm=raw_test.loc[cand.i, "comorbidity_count"].values)
        cand = cand.sort_values("cm", ascending=False)
    idx = int(cand.iloc[0]["i"]) if len(cand) else int(np.argsort(np.abs(y_prob - 0.35))[0])

print(f"Patient row {idx} | predicted risk {y_prob[idx]:.1%} | actual outcome "
      f"{'poor' if y_test.iloc[idx] == 1 else 'good'}")
print()
for c in ["age_band_grouped", "sex", "oks_t0_score", "comorbidity_count",
          "t0_symptom_period", "region"]:
    if c in raw_test.columns:
        print(f"  {c:22s} {raw_test.iloc[idx][c]}")

In [ ]:
local = ebm.explain_local(X_test.iloc[[idx]], y_test.iloc[[idx]])
d = local.data(0)
names, scores = np.array(d["names"], dtype=object), np.array(d["scores"], dtype=float)

k = 12
top = np.argsort(np.abs(scores))[::-1][:k][::-1]

fig, ax = plt.subplots(figsize=(9, 6))
colours = ["#C44E52" if scores[i] > 0 else "#4C78A8" for i in top]
ax.barh([str(names[i]) for i in top], scores[top], color=colours)
ax.axvline(0, color="#333", lw=1.2)
ax.set_xlabel("Contribution to log-odds of a poor outcome")
ax.set_title(f"Why this patient scored {y_prob[idx]:.0%}", fontweight="bold")
ax.grid(alpha=.25, axis="x")
plt.tight_layout(); plt.show()

intercept = float(np.asarray(ebm.intercept_).ravel()[0])
total = intercept + scores.sum()
print(f"intercept {intercept:+.4f}")
print(f"contributions {scores.sum():+.4f}")
print(f"sum (log-odds) {total:+.4f}  ->  probability {1/(1+np.exp(-total)):.1%}")
print("\nThat arithmetic is the model. No approximation anywhere in it.")

## 4. Six patients — where the model is right and where it is wrong

From `07_ebm_explainability`. Three the model got right and three it got wrong, side by side.

Showing failures on stage is worth more than showing successes. It is what makes the case
that this is decision *support*: a clinician who can see the model's reasoning can also see
when the reasoning does not fit the patient in front of them, and overrule it.

In [ ]:
threshold = float(mv.tags.get("operating_threshold", 0.5))
y_hat = (y_prob >= threshold).astype(int)
correct = y_hat == np.asarray(y_test)

pool_right = np.where(correct & (np.asarray(y_test) == 1))[0]      # caught poor outcomes
pool_wrong = np.where(~correct & (np.asarray(y_test) == 1))[0]     # missed poor outcomes

rng = np.random.default_rng(42)
picks = [(int(i), "caught")  for i in rng.choice(pool_right, min(3, len(pool_right)), replace=False)] + \
        [(int(i), "missed")  for i in rng.choice(pool_wrong, min(3, len(pool_wrong)), replace=False)]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
rows = []
for ax, (i, kind) in zip(axes.ravel(), picks):
    d = ebm.explain_local(X_test.iloc[[i]], y_test.iloc[[i]]).data(0)
    s = np.array(d["scores"], dtype=float)
    n = np.array(d["names"], dtype=object)
    t = np.argsort(np.abs(s))[::-1][:6][::-1]

    ax.barh([str(n[j])[:28] for j in t], s[t],
            color=["#C44E52" if s[j] > 0 else "#4C78A8" for j in t])
    ax.axvline(0, color="#333", lw=1)
    ax.set_title(f"{kind} · risk {y_prob[i]:.0%} · actual poor", fontsize=10,
                 color="#1D6F42" if kind == "caught" else "#9E2B25", fontweight="bold")
    ax.tick_params(labelsize=8)

    rows.append({"row": i, "outcome": kind, "predicted_risk": round(float(y_prob[i]), 3),
                 "oks_t0": raw_test.iloc[i].get("oks_t0_score"),
                 "comorbidities": raw_test.iloc[i].get("comorbidity_count"),
                 "top_driver": str(n[np.argmax(np.abs(s))])})

plt.tight_layout(); plt.show()
display(pd.DataFrame(rows))

print("The missed patients are the interesting ones. If their profiles look benign, the "
      "model is behaving reasonably and the information simply is not in the data — which "
      "is a limitation for the model card, not a bug.")

## 5. The black-box contrast — SHAP on a random forest

Now the other side. We fit the tuned random forest, explain it with SHAP, and put the two
approaches next to each other.

SHAP is genuinely good work — TreeSHAP gives exact Shapley values for tree ensembles. The
distinction is not accuracy, it is **provenance**: the EBM's numbers come from the model's
own structure, while SHAP's come from a post-hoc attribution over that structure. In a
regulated setting you have to explain that difference, and it is easier to explain when
both are on screen.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import shap

# Parameters from 41_model_search; replace with whatever the search returned.
rf = Pipeline([
    ("scale", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=300, max_depth=15, min_samples_leaf=4,
                                     min_samples_split=10, max_features="sqrt",
                                     class_weight="balanced_subsample",
                                     random_state=42, n_jobs=-1)),
])
rf.fit(X_train, y_train)

X_test_scaled = pd.DataFrame(rf.named_steps["scale"].transform(X_test), columns=X_test.columns)

rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X_test_scaled), size=min(200, len(X_test_scaled)), replace=False)
X_shap = X_test_scaled.iloc[sample_idx]

explainer = shap.TreeExplainer(rf.named_steps["model"])
_sv = explainer.shap_values(X_shap, check_additivity=False)
_ev = explainer.expected_value

# TreeExplainer returns three different shapes depending on version and model.
_ev = np.ravel(_ev)
if isinstance(_sv, list):                 # older shap: [class0, class1]
    sv, ev = _sv[1], float(_ev[-1])
elif np.ndim(_sv) == 3:                   # shap >= 0.45: (n, features, classes)
    sv, ev = np.asarray(_sv)[:, :, 1], float(_ev[-1])
else:                                     # single-output
    sv, ev = np.asarray(_sv), float(_ev[0])

sv = np.asarray(sv, dtype="float64")
assert sv.shape == X_shap.shape, f"got {sv.shape}, expected {X_shap.shape}"

# base_values must be one value PER ROW, not a scalar. shap's Explanation
# arithmetic (.abs.mean(0), used by bar/beeswarm/scatter) calls
# base_values.shape, and a float does not have one.
shap_values = shap.Explanation(
    values=sv,
    base_values=np.full(sv.shape[0], ev, dtype="float64"),
    data=X_shap.values,
    feature_names=list(X_shap.columns),
)
print(f"SHAP values: {shap_values.values.shape}  base {shap_values.base_values.shape}  "
      f"(expect ({len(X_shap)}, {X_test.shape[1]}))")


In [ ]:
print("Global importance — mean |SHAP| across the sample")
shap.plots.bar(shap_values, max_display=15, show=True)

print("\nBeeswarm — every dot is one patient")
shap.plots.beeswarm(shap_values, max_display=15, show=True)

In [ ]:
# Dependence plots for the top 3, which is where SHAP genuinely adds something the
# EBM shape functions do not: the colour reveals interaction effects.
mean_abs = np.abs(shap_values.values).mean(axis=0)
top3 = [X_shap.columns[i] for i in np.argsort(mean_abs)[::-1][:3]]

for feat in top3:
    print(f"\nDependence: {feat}")
    shap.plots.scatter(shap_values[:, feat], color=shap_values, show=True)

## 6. Side by side — do they agree?

If the glassbox and the black box rank features the same way, that is a real finding: the
interpretable model is not buying its readability by seeing something different. If they
disagree sharply, that needs explaining before either goes near a patient.

In [ ]:
ebm_main = {term_names[i]: importances[i]
            for i in range(len(term_names)) if len(ebm.term_features_[i]) == 1}
shap_imp = dict(zip(X_shap.columns, mean_abs))

common = [f for f in X_test.columns if f in ebm_main and f in shap_imp]
cmp = pd.DataFrame({
    "feature":  common,
    "ebm":      [ebm_main[f] for f in common],
    "shap_rf":  [shap_imp[f] for f in common],
})
cmp["ebm_rank"]  = cmp["ebm"].rank(ascending=False).astype(int)
cmp["shap_rank"] = cmp["shap_rf"].rank(ascending=False).astype(int)
cmp["rank_gap"]  = (cmp["ebm_rank"] - cmp["shap_rank"]).abs()
cmp = cmp.sort_values("ebm_rank")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
top = cmp.head(15).iloc[::-1]
ypos = np.arange(len(top))
ax.barh(ypos - .2, top["ebm"] / top["ebm"].max(), height=.4, label="EBM (own terms)", color="#4C78A8")
ax.barh(ypos + .2, top["shap_rf"] / top["shap_rf"].max(), height=.4, label="SHAP on RF", color="#C44E52")
ax.set_yticks(ypos); ax.set_yticklabels(top["feature"], fontsize=8)
ax.set_xlabel("Importance (scaled to each model's maximum)")
ax.set_title("Same features, two methods", fontweight="bold")
ax.legend()

ax = axes[1]
ax.scatter(cmp["ebm_rank"], cmp["shap_rank"], s=28, color="#4C78A8", alpha=.75)
lim = max(cmp["ebm_rank"].max(), cmp["shap_rank"].max())
ax.plot([1, lim], [1, lim], "--", color="#888", lw=1)
for _, r in cmp[cmp["rank_gap"] >= 10].iterrows():
    ax.annotate(r["feature"][:22], (r["ebm_rank"], r["shap_rank"]), fontsize=7,
                xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("EBM rank"); ax.set_ylabel("SHAP-on-RF rank")
ax.set_title("Agreement (on the diagonal = identical ranking)", fontweight="bold")
ax.grid(alpha=.3)

plt.tight_layout(); plt.show()

rho = cmp["ebm_rank"].corr(cmp["shap_rank"], method="spearman")
print(f"Spearman rank correlation: {rho:.3f}")
print("\nBiggest disagreements:")
print(cmp.nlargest(5, "rank_gap")[["feature", "ebm_rank", "shap_rank", "rank_gap"]].to_string(index=False))

## 7. Log the figures as governance artefacts

Explanations are evidence. Logging them to MLflow ties this analysis to a specific model
version, so "we explained the model" has a date, a version, and an artefact behind it rather
than being a claim in a slide deck.

In [ ]:
import tempfile, os

with mlflow.start_run(run_name="explainability") as run:
    mlflow.set_tags({
        "run_type":         "explainability",
        "notebook":         "43_explainability",
        "model_name":       REGISTERED_MODEL,
        "model_version":    str(mv.version),
        "risk_class":       "high_risk_eu_ai_act",
        "gdpr_article_22":  "local_explanations_available",
    })
    mlflow.log_metric("ebm_shap_rank_correlation", float(rho))
    mlflow.log_metric("n_ebm_terms", len(term_names))
    mlflow.log_metric("n_interactions", sum(1 for t in ebm.term_features_ if len(t) > 1))

    with tempfile.TemporaryDirectory() as d:
        p = os.path.join(d, "ebm_term_importance.csv")
        pd.DataFrame({"term": term_names, "importance": importances}) \
          .sort_values("importance", ascending=False).to_csv(p, index=False)
        mlflow.log_artifact(p, "explainability")

        p = os.path.join(d, "ebm_vs_shap_ranking.csv")
        cmp.to_csv(p, index=False)
        mlflow.log_artifact(p, "explainability")

    print(f"Logged to run {run.info.run_id}")

MlflowClient().set_model_version_tag(
    REGISTERED_MODEL, mv.version, "explainability_reviewed", "true")
MlflowClient().set_model_version_tag(
    REGISTERED_MODEL, mv.version, "ebm_shap_rank_correlation", f"{rho:.3f}")
print("Model version tagged.")

## 8. The line to land in segment 4

> A regulator can read the shape functions. A clinician can challenge them. A black box
> offers neither — and under the AI Act, "the model is more accurate" is not an answer to
> "why was this patient told that?"

Have these three numbers to hand:

* the **accuracy cost** of choosing the glassbox model (from `41_model_search` §7),
* the **rank correlation** between the EBM's own terms and SHAP on the random forest —
  printed above,
* **Margaret's contributions summing to her score** — the arithmetic in §3.

The third is the one that lands. Everything else is supporting evidence.

---

**Next:** `50_batch_score` writes these explanations for the whole cohort so the app and the
Data Agent can read them.